# Fiat-Shamir: de protocolo interactivo a firma no interactiva

**Trabajo Final — Programación Científica 2026-1**

Integrantes:\
Delgado Ortiz, David \
Fonseca Aldana, Miguel Angel\
Moreno Ceballos, Jose Daniel\
Ospina Ocampo, Juan Diego\
Urrutia Manyoma, Haison



## 1. Introducción y motivación

_(2-3 frases: por qué eligieron este tema, si ya lo trabajaron en Criptografía y qué esperan mejorar con este curso)_


## 2. Contexto teórico

### 2.1 Protocolo interactivo de conocimiento (base: raíces cuadradas mod N)

El protocolo de **Fiat-Shamir** permite a un **Prover** convencer a un **Verifier** de que conoce un secreto \(s\) sin revelarlo.

Originalmente, es un protocolo **interactivo** que consta de tres pasos:

1. **Compromiso (Commitment):**

   $$
   x = r^2 \pmod{N}
   $$

2. **Reto (Challenge):**

   $$
   e \in \{0,1\}
   $$

3. **Respuesta (Response):**

   $$
   y = r \cdot s^e \pmod{N}
   $$

### 2.2 La transformación de Fiat-Shamir (1986)

En 1986, **Amos Fiat** y **Adi Shamir** propusieron una transformación brillante: reemplazar al **Verifier** generando el reto mediante una función hash criptográfica aplicada al compromiso.

En lugar de que el **Verifier** elija el reto, éste se calcula como

$$
e = H(x),
$$

donde \(H\) es una función hash criptográfica.

Esta transformación elimina la necesidad de interacción entre el **Prover** y el **Verifier**, convirtiendo el protocolo de identificación en un **esquema de firma digital**.

La seguridad de esta transformación depende de modelar la función hash como un **oráculo aleatorio (Random Oracle)**, de modo que el reto sea impredecible y no pueda ser manipulado por el **Prover**.

### 2.3 Zero-knowledge vs. Soundness

_(las dos definiciones trabajadas: qué garantiza cada una y a quién protege)_


In [1]:
# Imports generales
import numpy as np
import matplotlib.pyplot as plt
import hashlib
import time
import random
import math
from sympy import randprime

# Nota: Usamos `sympy.randprime` para generar números primos grandes de forma eficiente.
# La aritmética modular (potencias) usará la función nativa de Python `pow(base, exp, mod)`,
# la cual está implementada en C y es altamente optimizada (usa exponenciación binaria).

## 3. Implementación del protocolo Fiat-Shamir

### 3.1 Configuración: generación de N = p·q y del secreto

_(código: elegir p, q primos, calcular N, generar secreto s y v = s^2 mod N)_


In [2]:
# Función para generar los parámetros públicos y el secreto
def generar_parametros(bits=256):
    """
    Genera N = p * q, un secreto 's' y la clave pública 'v = s^2 mod N'.
    """
    # Generamos dos primos aleatorios de aproximadamente 'bits/2' bits cada uno
    half_bits = bits // 2
    p = randprime(2**(half_bits-1), 2**half_bits)
    q = randprime(2**(half_bits-1), 2**half_bits)

    # Asegurarnos de que p y q sean distintos
    while p == q:
        q = randprime(2**(half_bits-1), 2**half_bits)

    N = p * q

    # El secreto 's' debe ser coprimo con N
    s = random.randint(2, N - 2)
    while math.gcd(s, N) != 1:
        s = random.randint(2, N - 2)

    # Clave pública v = s^2 mod N
    v = pow(s, 2, N)

    return {'p': p, 'q': q, 'N': N, 's': s, 'v': v}


### 3.2 Prover: compromiso y respuesta


In [3]:
# Función que genera el compromiso (r, x = r^2 mod N)
def generar_compromiso(N):
    """
    El Prover elige un número aleatorio 'r' y se compromete con 'x = r^2 mod N'.
    """
    r = random.randint(1, N - 1)
    x = pow(r, 2, N)
    return r, x

### 3.3 El reto vía hash (la transformación de Fiat-Shamir)


In [5]:
# Función que calcula el reto como hash(x) en vez de un valor aleatorio del verifier
def calcular_reto_fiat_shamir(x):
    """
    Transformación de Fiat-Shamir:
    En lugar de que el Verifier elija el reto 'e', se calcula como un hash del compromiso 'x'.
    Esto convierte el protocolo interactivo en uno no interactivo.
    Retornamos 'e' como un bit (0 o 1) que es el estándar del protocolo básico de Fiat-Shamir.
    """
    # Hasseamos el compromiso (convertido a string)
    hash_hex = hashlib.sha256(str(x).encode()).hexdigest()
    # Convertimos el hash a entero y tomamos el módulo 2 para obtener un bit (0 o 1)
    e = int(hash_hex, 16) % 2
    return e

### 3.4 Verificación


In [ ]:
# TODO: función que verifica la prueba completa (compromiso, reto, respuesta)
def verificar_prueba(x, e, y, v, N):
    pass


## 4. Ejemplo aplicado — Parte A: costo computacional

**Pregunta:** ¿cómo escala el costo de generar y verificar una prueba en función
del tamaño de N (256, 512, 1024, 2048 bits)?

_(medir tiempos de exponenciación modular / generación de parámetros vs. tamaño de N,
graficar en escala log-log)_


In [ ]:
# TODO: medir tiempos para distintos tamaños de N
tamanos_bits = [256, 512, 1024, 2048]
tiempos = []

# for bits in tamanos_bits:
#     t0 = time.time()
#     ...
#     tiempos.append(time.time() - t0)


In [ ]:
# TODO: graficar tiempo vs tamaño de N (log-log)
plt.figure(figsize=(6,4))
# plt.plot(tamanos_bits, tiempos, marker='o')
plt.xlabel("Tamaño de N (bits)")
plt.ylabel("Tiempo (s)")
plt.title("Costo computacional de Fiat-Shamir vs. tamaño de N")
plt.show()


## 5. Ejemplo aplicado — Parte B: análisis estadístico del hash (Monte Carlo)

**Pregunta:** ¿el hash se comporta como un generador de retos uniforme e impredecible
(modelo de oráculo aleatorio)? ¿Qué pasa si tuviera sesgos?

_(simular miles de compromisos aleatorios, calcular hash(compromiso), analizar la
distribución de los retos resultantes: uniformidad, colisiones, etc.)_


In [ ]:
# TODO: simulación Monte Carlo de retos generados por hash
n_simulaciones = 100_000
retos = []

# for _ in range(n_simulaciones):
#     compromiso = random.randint(0, 2**256)
#     reto = int(hashlib.sha256(str(compromiso).encode()).hexdigest(), 16)
#     retos.append(reto)


In [ ]:
# TODO: graficar distribución de los retos (histograma) y comentar uniformidad
plt.figure(figsize=(6,4))
# plt.hist(retos, bins=50)
plt.xlabel("Valor del reto")
plt.ylabel("Frecuencia")
plt.title("Distribución de retos generados por hash (Monte Carlo)")
plt.show()


## 6. Conclusiones

_(síntesis breve: qué muestran las partes A y B, y cómo conectan con la seguridad
y eficiencia real de Fiat-Shamir)_


## 7. Opinión del grupo

_(cada integrante escribe su propia opinión sobre el tema y autoevaluación del curso —
esto va también en el informe PDF, aquí puede quedar como referencia)_
